# 08 · M4GT external validation: Falcon dual-model scoring

This notebook runs the external validation on the frozen M4GT Subtask B manifest. It uses the Falcon-7B-Instruct / Falcon-7B pair and the Fast-DetectGPT and Binoculars formulas of `07_falcon_dual_scoring.ipynb`.

Prerequisites: `DATA_DIR` is prepared as described in data/README.md (including external/m4gt/SubtaskB.jsonl), and the GPU can hold two 7B models at once. No AUROC or FPR conclusion exists before this notebook has run.

**How to run**: pick a Colab server with an **A100 GPU** and click "Run All"; about 20 minutes. To rerun, first use Runtime → Disconnect and delete runtime, then Run All; otherwise the models loaded last time still occupy GPU memory and the run crashes. Section 4 prints progress and elapsed minutes every 50 batches; changing numbers mean it is still running.


## Method and locked conditions

- Raw data: `external/m4gt/SubtaskB.jsonl`
- Frozen manifest: `manifests/m4gt_subtaskb_manifest.csv`
- Raw file SHA-256: `4d65fc1fb93b5c9d21b226aca06b610ee031fa9ccbc18fa6f39d16273dd281c7`
- Manifest SHA-256: `eba1e8591506dfd941a4c2fa8cef71a830026326c7a27968a61f10b82e2bcacf`
- Human label: `label == 1`; every other label counts as AI.

Output is written only to `08_m4gt_natural_text/`; the raw data and the manifest are never overwritten.

In [ ]:
# Same as 07_falcon_dual_scoring.ipynb. Falcon does not need sentencepiece; scikit-learn is the Colab default. The actual versions are recorded in run_info.json at the end.
%pip install -q -U transformers huggingface-hub accelerate


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/curvature-margin/data')  # change if your data folder differs
assert PROJECT_DIR.exists(), f'Not found: {PROJECT_DIR}'
print(PROJECT_DIR)

In [ ]:
import csv
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import roc_auc_score

RAW_PATH = PROJECT_DIR / 'external/m4gt/SubtaskB.jsonl'
MANIFEST_PATH = PROJECT_DIR / 'manifests/m4gt_subtaskb_manifest.csv'
OUTPUT_DIR = PROJECT_DIR / '08_m4gt_natural_text'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_RAW_SHA256 = '4d65fc1fb93b5c9d21b226aca06b610ee031fa9ccbc18fa6f39d16273dd281c7'
EXPECTED_MANIFEST_SHA256 = 'eba1e8591506dfd941a4c2fa8cef71a830026326c7a27968a61f10b82e2bcacf'
SCORER = 'tiiuae/falcon-7b-instruct'
OBSERVER = 'tiiuae/falcon-7b'
MAX_TOKENS = 512
BATCH_SIZE = 8  # as in 07; lower it if memory runs out
SEED = 20260926

assert torch.cuda.is_available(), 'A GPU runtime is required; choose a GPU in Colab and rerun from this cell.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, 'transformers:', transformers.__version__)

## 1. Verify the raw file and the frozen manifest

Stop if either hash does not match: the data or the manifest then differs from the version locked in the study design.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

assert sha256_file(RAW_PATH) == EXPECTED_RAW_SHA256, 'Raw file SHA-256 does not match.'
assert sha256_file(MANIFEST_PATH) == EXPECTED_MANIFEST_SHA256, 'Manifest SHA-256 does not match.'
manifest = pd.read_csv(MANIFEST_PATH)
assert len(manifest) == 8400, f'Expected 8400 rows, got {len(manifest)}.'
assert set(manifest.columns) == {'line_number', 'source', 'label', 'model', 'text_chars', 'text_sha256'}
display(manifest.groupby(['source', 'label', 'model']).size().rename('n').reset_index().head(12))

## 2. Load the texts listed in the manifest

Only the raw lines named in the manifest are read. The SHA-256 of every text is checked again so that line numbers and contents stay in sync.

In [ ]:
needed = {int(row.line_number): row for row in manifest.itertuples(index=False)}
records = []
with RAW_PATH.open(encoding='utf-8') as handle:
    for line_number, line in enumerate(handle, start=1):
        row = needed.get(line_number)
        if row is None:
            continue
        item = json.loads(line)
        assert hashlib.sha256(item['text'].encode('utf-8')).hexdigest() == row.text_sha256
        assert item['source'] == row.source and int(item['label']) == int(row.label) and item['model'] == row.model
        records.append({
            'line_number': line_number, 'source': row.source, 'label': int(row.label),
            'model': row.model, 'text_sha256': row.text_sha256, 'text': item['text'],
        })

data = pd.DataFrame(records).sort_values('line_number').reset_index(drop=True)
assert len(data) == len(manifest)
data['who'] = np.where(data.label.eq(1), 'human', 'ai')
display(data.groupby(['source', 'who', 'model']).size().rename('n').reset_index().head(12))

## 3. Load the Falcon models

This setting needs a GPU that holds two 7B models. If memory runs out, lower `BATCH_SIZE`; do not change the models, the tokenizer or the maximum length and then report the results together with the main experiment.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc
for _name in ('scorer_model', 'observer_model'):  # on a rerun of this cell, free the models loaded last time, or the GPU runs out of memory
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(SCORER)
tokenizer.padding_side = 'right'
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
scorer_model = AutoModelForCausalLM.from_pretrained(SCORER, dtype=torch.bfloat16).cuda().eval()
observer_model = AutoModelForCausalLM.from_pretrained(OBSERVER, dtype=torch.bfloat16).cuda().eval()
print('models loaded')
print(f'GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GB')


## 4. Score every text

Definitions match the main experiment: `FastDetectGPT = (-Σ surprisal - Σ X) / sqrt(Σ XV)`; `Binoculars = -(mean(surprisal) / -mean(X))`. Higher scores mean more AI-like.

In [ ]:
data['input_token_count'] = data.text.map(lambda text: len(tokenizer(text, truncation=True, max_length=MAX_TOKENS)['input_ids']))
excluded_short = data.loc[data.input_token_count.lt(11)].drop(columns='text')
excluded_short.to_csv(OUTPUT_DIR / 'm4gt_excluded_short_texts.csv', index=False)
eligible = data.loc[data.input_token_count.ge(11)].copy()
assert len(eligible) > 0, 'No text is long enough to score.'
print(f'eligible={len(eligible)}; excluded_short={len(excluded_short)}')

@torch.no_grad()
def score_batch(texts):
    encoded = [tokenizer(text, truncation=True, max_length=MAX_TOKENS)['input_ids'] for text in texts]
    assert min(map(len, encoded)) >= 11
    length = max(map(len, encoded))
    input_ids = torch.full((len(encoded), length), pad_id, dtype=torch.long, device='cuda')
    attention_mask = torch.zeros((len(encoded), length), dtype=torch.long, device='cuda')
    for i, ids in enumerate(encoded):
        input_ids[i, :len(ids)] = torch.tensor(ids, device='cuda')
        attention_mask[i, :len(ids)] = 1
    logits_p = scorer_model(input_ids=input_ids, attention_mask=attention_mask).logits
    logits_q = observer_model(input_ids=input_ids, attention_mask=attention_mask).logits
    scores = []
    for i, ids in enumerate(encoded):
        n = len(ids)
        logp = torch.log_softmax(logits_p[i, :n - 1].float(), dim=-1)
        logq = torch.log_softmax(logits_q[i, :n - 1].float(), dim=-1)
        target = torch.tensor(ids[1:], device='cuda')
        surprisal = -logp.gather(1, target[:, None])[:, 0]
        p = logp.exp()
        q = logq.exp()
        expected_logp = (q * logp).sum(-1)
        variance = (q * logp * logp).sum(-1) - expected_logp.square()
        fdg = (-surprisal.sum() - expected_logp.sum()) / torch.sqrt(variance.sum())
        bino = -(surprisal.mean() / -expected_logp.mean())
        scores.append({
            'token_count': n - 1,
            'surprisal': float(surprisal.mean().cpu()),
            'curvature': float((-surprisal - expected_logp).mean().cpu()),
            'fast_detect_gpt': float(fdg.cpu()),
            'binoculars': float(bino.cpu()),
        })
    return scores

ordered = eligible.assign(_length=eligible.text.str.len()).sort_values('_length').reset_index()
import time
scored, t0 = [], time.time()
for start in range(0, len(ordered), BATCH_SIZE):
    batch = ordered.iloc[start:start + BATCH_SIZE]
    result = score_batch(batch.text.tolist())
    assert len(result) == len(batch)
    for original_index, values in zip(batch['index'], result):
        scored.append({'original_index': int(original_index), **values})
    if (start // BATCH_SIZE) % 50 == 0:
        print(f'  {start}/{len(ordered)} {(time.time() - t0) / 60:.1f} min', flush=True)

score_frame = pd.DataFrame(scored).set_index('original_index').sort_index()
results = pd.concat([eligible, score_frame], axis=1)
assert len(results) == len(eligible) and results[['fast_detect_gpt', 'binoculars']].notna().all().all()
score_output = results.drop(columns='text')
score_output.to_parquet(OUTPUT_DIR / 'm4gt_falcon_scores.parquet', index=False)
score_output.to_csv(OUTPUT_DIR / 'm4gt_falcon_scores.csv', index=False)
print('saved sample-level scores')

## 5. AUROC and FPR operating points

Same-data FPR = 1% sets the 99th-percentile threshold on all human texts and then estimates the AI TPR. The 50/50 split sets the threshold on the human texts with an even hash only and measures the realized FPR on the other half; it shows how unstable a threshold is when carried over to new data.

In [ ]:
def metrics_for_group(frame, detector):
    human = frame.loc[frame.who.eq('human'), detector].to_numpy()
    ai = frame.loc[frame.who.eq('ai'), detector].to_numpy()
    if len(human) == 0 or len(ai) == 0:
        return None
    threshold = np.quantile(human, 0.99, method='higher')
    calibration = frame.loc[frame.who.eq('human') & frame.text_sha256.str.slice(0, 8).map(lambda x: int(x, 16) % 2 == 0), detector].to_numpy()
    evaluation = frame.loc[frame.who.eq('human') & frame.text_sha256.str.slice(0, 8).map(lambda x: int(x, 16) % 2 == 1), detector].to_numpy()
    split_threshold = np.quantile(calibration, 0.99, method='higher')
    return {
        'detector': detector, 'n_human': len(human), 'n_ai': len(ai),
        'auroc': roc_auc_score(np.r_[np.zeros(len(human)), np.ones(len(ai))], np.r_[human, ai]),
        'same_data_fpr': float((human >= threshold).mean()),
        'same_data_tpr_at_1pct_fpr': float((ai >= threshold).mean()),
        'split_calibration_n': len(calibration), 'split_evaluation_n': len(evaluation),
        'split_actual_fpr': float((evaluation >= split_threshold).mean()),
        'split_tpr': float((ai >= split_threshold).mean()),
    }

summary_rows = []
for scope, frame in [('overall', results), *[(f'domain:{source}', g) for source, g in results.groupby('source')]]:
    for detector in ('fast_detect_gpt', 'binoculars'):
        summary_rows.append({'scope': scope, **metrics_for_group(frame, detector)})
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / 'm4gt_falcon_summary.csv', index=False)
display(summary.round(4))

## 6. Write traceable run information

Afterwards, copy the values of `m4gt_falcon_summary.csv` into the study documents. M4GT external results may be reported only after this cell has written its file.

In [ ]:
run_info = {
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'raw_sha256': sha256_file(RAW_PATH),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'scorer_model': SCORER,
    'observer_model': OBSERVER,
    'max_tokens': MAX_TOKENS,
    'batch_size': BATCH_SIZE,
    'seed': SEED,
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'gpu': torch.cuda.get_device_name(0),
}
(OUTPUT_DIR / 'm4gt_falcon_run_info.json').write_text(json.dumps(run_info, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('outputs:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)